# Testing Different Coordinate Systems for Location Encoder Validation

This notebook demonstrates three coordinate systems for GeoShapley + location encoder experiments:
1. **Grid coordinates**: Current validation approach (controlled)
2. **Regional geographic**: Realistic local studies (e.g., Chicago)
3. **Global geographic**: Earth-scale studies with proper lat/lon

**Key Question**: Should location encoders use grid indices or real geographic coordinates?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from spatial_dgp_utils import SpatialDGP, create_mgwr_compatible_data
import warnings
warnings.filterwarnings('ignore')

## 1. Grid Coordinates (Current Approach)

**Pros**: 
- Simple, controlled
- Matches original GeoShapley validation
- Fast debugging

**Cons**:
- Not realistic for location encoders
- Encoder `extent` parameter is arbitrary

In [ ]:
# Generate grid data (like current validation notebook)
dgp_grid = SpatialDGP(coord_system='grid', size=25)
data_grid = dgp_grid.generate_data(dgp_type='mgwr_style', random_seed=222)

print(f"Coordinate system: Grid")
print(f"Data shape: {data_grid['X'].shape}")
print(f"Coordinates range: {data_grid['coords'].min(axis=0)} to {data_grid['coords'].max(axis=0)}")
print(f"Extent for encoder: {data_grid['extent']}")
print(f"\nFirst 5 rows:")
print(data_grid['X'].head())

In [ ]:
# Visualize grid coordinates and coefficients
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

size = dgp_grid.size
coeffs = dgp_grid.generate_dgp_coefficients('mgwr_style')

for i, (name, coeff) in enumerate([('b0 (Intercept)', coeffs['b0']), 
                                     ('b1 (Coeff for X1)', coeffs['b1']),
                                     ('b2 (Coeff for X2)', coeffs['b2'])]):
    im = axes[i].imshow(coeff.reshape(size, size), cmap='viridis', origin='lower')
    axes[i].set_title(name)
    plt.colorbar(im, ax=axes[i])
    axes[i].set_xlabel('x_coord')
    axes[i].set_ylabel('y_coord')

plt.suptitle('Grid Coordinates: True Spatially Varying Coefficients', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Regional Geographic Coordinates (Recommended for Most Studies)

**Pros**:
- Real lat/lon coordinates
- Encoders behave as designed
- Publishable and interpretable
- Can specify real-world regions

**Example**: Chicago metro area (~100km × 100km)

In [ ]:
# Generate regional geographic data (Chicago)
dgp_regional = SpatialDGP(
    coord_system='regional',
    size=25,
    center_coords=(-87.65, 41.85),  # Chicago
    km_span=100  # 100km x 100km region
)

data_regional = dgp_regional.generate_data(dgp_type='mgwr_style', random_seed=222)

print(f"Coordinate system: Regional Geographic (Chicago)")
print(f"Data shape: {data_regional['X'].shape}")
print(f"Lon range: {data_regional['coords'][:, 0].min():.2f} to {data_regional['coords'][:, 0].max():.2f}")
print(f"Lat range: {data_regional['coords'][:, 1].min():.2f} to {data_regional['coords'][:, 1].max():.2f}")
print(f"Extent for encoder: {data_regional['extent']}")
print(f"\nFirst 5 rows:")
print(data_regional['X'].head())

In [ ]:
# Visualize regional geographic data
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

size = dgp_regional.size
coeffs_regional = dgp_regional.generate_dgp_coefficients('mgwr_style')

for i, (name, coeff) in enumerate([('b0 (Intercept)', coeffs_regional['b0']), 
                                     ('b1 (Coeff for X1)', coeffs_regional['b1']),
                                     ('b2 (Coeff for X2)', coeffs_regional['b2'])]):
    im = axes[i].imshow(coeff.reshape(size, size), cmap='viridis', origin='lower')
    axes[i].set_title(name)
    plt.colorbar(im, ax=axes[i])
    axes[i].set_xlabel('Longitude')
    axes[i].set_ylabel('Latitude')

plt.suptitle('Regional Geographic (Chicago): True Spatially Varying Coefficients', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Show spatial distribution of points on a map-like view
plt.figure(figsize=(8, 6))
plt.scatter(data_regional['coords'][:, 0], data_regional['coords'][:, 1], 
           c=data_regional['y'], cmap='RdYlBu_r', s=50, alpha=0.6)
plt.colorbar(label='y (outcome)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Regional Geographic Data: Spatial Distribution of Outcome')
plt.grid(alpha=0.3)
plt.axis('equal')
plt.tight_layout()
plt.show()

print(f"This represents a {dgp_regional.kwargs.get('km_span', 100)}km × {dgp_regional.kwargs.get('km_span', 100)}km region")
print(f"Center: {dgp_regional.kwargs.get('center_coords', 'N/A')}")

## 3. Global Geographic Coordinates (For Large-Scale Studies)

**Pros**:
- Tests encoder on full Earth surface
- Realistic for climate, species, satellite data
- Can include latitude effects

**Cons**:
- Need to handle spherical geometry
- More complex DGPs

In [ ]:
# Generate global geographic data
dgp_global = SpatialDGP(
    coord_system='global',
    size=25,  # 25x25 = 625 points globally
    sampling='stratified'
)

data_global = dgp_global.generate_data(dgp_type='latitude', random_seed=222)

print(f"Coordinate system: Global Geographic")
print(f"Data shape: {data_global['X'].shape}")
print(f"Lon range: {data_global['coords'][:, 0].min():.2f} to {data_global['coords'][:, 0].max():.2f}")
print(f"Lat range: {data_global['coords'][:, 1].min():.2f} to {data_global['coords'][:, 1].max():.2f}")
print(f"Extent for encoder: {data_global['extent']}")
print(f"\nFirst 5 rows:")
print(data_global['X'].head())

In [ ]:
# Visualize global data on a world map-like projection
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

coeffs_global = dgp_global.generate_dgp_coefficients('latitude')

# Plot as scatter (simulating world map)
for i, (ax, name, coeff) in enumerate([
    (axes[0, 0], 'b0 (Intercept)', coeffs_global['b0']),
    (axes[0, 1], 'b1 (Coeff for X1)', coeffs_global['b1']),
    (axes[1, 0], 'b2 (Coeff for X2)', coeffs_global['b2']),
    (axes[1, 1], 'Outcome (y)', data_global['y'])
]):
    scatter = ax.scatter(data_global['coords'][:, 0], data_global['coords'][:, 1],
                        c=coeff, cmap='RdYlBu_r', s=30, alpha=0.7)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(name)
    ax.grid(alpha=0.3)
    ax.set_xlim(-180, 180)
    ax.set_ylim(-90, 90)
    plt.colorbar(scatter, ax=ax)

plt.suptitle('Global Geographic: Latitude-Dependent DGP', fontsize=16)
plt.tight_layout()
plt.show()

## Comparison Summary

In [ ]:
# Create comparison table
comparison = pd.DataFrame([
    {
        'Coordinate System': 'Grid',
        'Coord Range X': f"{data_grid['coords'][:, 0].min():.1f} to {data_grid['coords'][:, 0].max():.1f}",
        'Coord Range Y': f"{data_grid['coords'][:, 1].min():.1f} to {data_grid['coords'][:, 1].max():.1f}",
        'Extent': str(data_grid['extent']),
        'Use Case': 'Controlled validation, debugging',
        'Encoder Realistic': 'No'
    },
    {
        'Coordinate System': 'Regional (Chicago)',
        'Coord Range X': f"{data_regional['coords'][:, 0].min():.2f} to {data_regional['coords'][:, 0].max():.2f}",
        'Coord Range Y': f"{data_regional['coords'][:, 1].min():.2f} to {data_regional['coords'][:, 1].max():.2f}",
        'Extent': f"({data_regional['extent'][0]:.2f}, {data_regional['extent'][1]:.2f}, ...)",
        'Use Case': 'Realistic local studies, publishable',
        'Encoder Realistic': 'Yes'
    },
    {
        'Coordinate System': 'Global',
        'Coord Range X': f"{data_global['coords'][:, 0].min():.1f} to {data_global['coords'][:, 0].max():.1f}",
        'Coord Range Y': f"{data_global['coords'][:, 1].min():.1f} to {data_global['coords'][:, 1].max():.1f}",
        'Extent': '(-180, 180, -90, 90)',
        'Use Case': 'Earth-scale, climate, species',
        'Encoder Realistic': 'Yes'
    }
])

print("\n" + "="*80)
print("COMPARISON: Three Coordinate Systems")
print("="*80)
print(comparison.to_string(index=False))
print("="*80)

## Recommendations for Your Experiments

### Stage 1: Quick Validation (Use Grid)
- Test that GeoShapley + location encoders work correctly
- Fast iteration, simple debugging
- **Caveat**: Encoder `extent` parameter is arbitrary

### Stage 2: Main Experiments (Use Regional)
- **Recommended for most analyses**
- Real geographic coordinates → encoders behave as designed
- Can specify real regions (Chicago, California, etc.)
- Publishable and interpretable
- Easy to scale from 100km to 1000km regions

### Stage 3: Large-Scale (Use Global)
- For global-scale research questions
- Tests encoder performance across Earth's surface
- Can include latitude/longitude effects naturally

### How to Update Your Current Code

In `help_utils.py`, update `get_loc_embeddings` extent:
```python
# Instead of:
extent = (0, 200, 0, 200)  # arbitrary!

# Use:
extent = (-88.5, -86.8, 41.3, 42.4)  # Chicago regional
# OR:
extent = (-180, 180, -90, 90)  # Global
# OR pass as parameter from data generation
```

## Test: Can we reproduce current validation notebook with regional coords?

Let's create data similar to `mgwr_sim.csv` but with geographic coordinates:

In [ ]:
# Generate MGWR-compatible data with regional coordinates
df_regional_mgwr, extent_regional = create_mgwr_compatible_data(
    coord_system='regional',
    size=25,
    center_coords=(-87.65, 41.85),  # Chicago
    km_span=100
)

print("Generated data compatible with mgwr_sim.csv format:")
print(df_regional_mgwr.head())
print(f"\nShape: {df_regional_mgwr.shape}")
print(f"Columns: {list(df_regional_mgwr.columns)}")
print(f"\nExtent for location encoders: {extent_regional}")

# Save for use in other notebooks
# df_regional_mgwr.to_csv('./data/mgwr_sim_regional_chicago.csv', index=False)
print("\n✅ This data can now be used with location encoders!")

## Next Steps

1. **Update `help_utils.py`**:
   - Pass `extent` parameter to `get_loc_embeddings()` from data generation
   - Don't hardcode `extent = (0, 200, 0, 200)`

2. **Update `embeddingsRun.py`**:
   - Use `spatial_dgp_utils.py` to generate data with geographic coordinates
   - Pass correct extent to encoder initialization

3. **Create validation experiments**:
   - Run with regional coordinates (Chicago, NYC, etc.)
   - Compare encoder performance with proper geographic context
   - Check if amplitude issues persist with realistic coordinates

4. **Scale up**:
   - Test different region sizes (10km, 100km, 1000km)
   - Test global sampling for Earth-scale questions